# 1. Load and Pre check

In [35]:
import ipp
from autoregressx import *
# model_Eval = ipp.pd.DataFrame()
ipp.warnings.filterwarnings("ignore")
ipp.initial_Check()

In [ ]:
df = ipp.pd.read_csv('/Users/nikhilprao/Documents/Data/Boston.csv', index_col=0)
df.reset_index(drop=True)

# 2. Preprocessing 

In [ ]:
# df.isnull().sum()
# # applying the method
# nan_in_df = df.isnull().sum().any()
 
# # Print the dataframe
# print(type(nan_in_df))

# df.info()

# df.describe()

### Ask User for predictive column

In [ ]:
# Display column names to the user
print("Available predictor variables:")
for idx, col in enumerate(df.columns):
    print(f"{idx + 1}. {col}")

# Ask the user to choose a predictor variable
selected_index = int(input("Enter the index of the predictor variable you want to choose: ")) - 1

# Validate user input
if 0 <= selected_index < len(df.columns):
    pattern = df.columns[selected_index]
    print(f"Selected predictor variable: {pattern}")
else:
    print("Invalid index selected. Please choose a valid index.")


predictor_variable = df.filter(regex=f'^{pattern}').columns[0]

## Model Mind prediction

In [ ]:
ipp.workflow(df, predictor_variable)

## 1. Raw model
### Needs change here

In [ ]:
ipp.interModel(df, predictor_variable, "raw")

## Pipeline for Outliers

In [ ]:
df_outlier_cleaned = ipp.main_outliers(df, predictor_variable)

In [ ]:
df_outlier_cleaned

In [ ]:
ipp.interModel(df_outlier_cleaned, predictor_variable, "outliers")

## Co relation matrix

In [ ]:
corr_mat=df_outlier_cleaned.corr()
# print(type(corr_mat))

In [ ]:
unique_counts = df.nunique()
print(unique_counts)
df_outlier_cleaned.info()

## Adding the threshold from corr matrix for model

In [ ]:
df_filtered, high_loss = ipp.remove_high_correlation_features(df_outlier_cleaned,predictor_variable)
print(high_loss)
ipp.update_high_correlation_features(high_loss["High_loss"])

In [ ]:
df_filtered.columns

In [ ]:
low_threshold_value = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]
results = {}

for i in low_threshold_value:
    df_filtered, low_loss = ipp.remove_low_correlation_features(df_outlier_cleaned, i, predictor_variable)
    num_features = len(low_loss["Low_loss"])
    
    results[i] = {
        "df_filtered": df_filtered,  # Stores the dataframe
        "num_features": num_features  # Stores the number of low-correlation features removed
    }

    model_name = "LTH_"
    ipp.interModel(df_filtered, predictor_variable, model_name+str(i))
    ipp.update_low_correlation_features(i, low_loss["Low_loss"])
    # print(df_filtered.head(2))
    # print(" ----------------- ")

# Return the dictionary containing all results
# results


In [ ]:
df_04 = results[0.4]["df_filtered"]
df_04.head(2)

## Skew Handling

In [ ]:
# Define skewness thresholds
high_skew_threshold = 1
moderate_skew_threshold = 0.5

# Calculate skewness for all columns
skew_values = df_04.skew()

# Categorize columns based on skewness values
highly_skewed = skew_values[abs(skew_values) > high_skew_threshold].index.tolist()
moderately_skewed = skew_values[(abs(skew_values) >= moderate_skew_threshold) & (abs(skew_values) <= high_skew_threshold)].index.tolist()
low_skew = skew_values[abs(skew_values) < moderate_skew_threshold].index.tolist()

# Print categorized columns
print("Highly Skewed:", highly_skewed)
print("Moderately Skewed:", moderately_skewed)
print("Low Skew:", low_skew)

In [ ]:
# Main function to apply skew handling
def skew_handling(df, highly_skewed, moderately_skewed):
    df = ipp.handle_high_skew(df, highly_skewed)  # Handling high skew
    ipp.interModel(df, predictor_variable, "High_skew")
    df = ipp.handle_moderate_skew(df, moderately_skewed)  # Handling moderate skew
    ipp.interModel(df, predictor_variable, "Moderate_skew")
    
    print("\n Skew Handling Completed. Returning Transformed DataFrame.")
    return df

df_skew = skew_handling(df_04, highly_skewed, moderately_skewed)

In [ ]:
try:
    with open(ipp.json_file_path, "r") as file:
        status_data = ipp.json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

# Now you can safely modify the status_data object
status_data["pre_processing"]["Skew"]["Low"]["handling"] = False
status_data["pre_processing"]["Skew"]["Low"]["features"] = low_skew

# Write the updated status back to the file
with open(ipp.json_file_path, "w") as file:
    ipp.json.dump(status_data, file, indent=4)

In [ ]:
df_outlier_cleaned.skew()

In [ ]:
# print(df_skew.head(2))
df_skew.skew()

In [ ]:
df_skew

In [ ]:
ipp.workflow(df_skew, predictor_variable)

In [ ]:
## Indpendent and dependent features
from sklearn.model_selection import train_test_split
X = df_skew.drop([predictor_variable], axis=1)
y = df_skew[predictor_variable]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# scaler_required_models = {
#     "LinearRegression": LinearRegression(),
#     "Ridge": Ridge(),
#     "Lasso": Lasso(),
#     "SVR": SVR(),
#     "KNeighbors": KNeighborsRegressor()
# }

# non_scaler_models = {
#     "DecisionTree": DecisionTreeRegressor(),
#     "RandomForest": RandomForestRegressor(),
#     "GradientBoosting": GradientBoostingRegressor()
# }

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# pipelines = {}

# # Scaler-sensitive models
# for name, model in scaler_required_models.items():
#     pipe = Pipeline([
#         ('scaler', StandardScaler()),
#         ('model', model)
#     ])
#     pipe.fit(X_train, y_train)
#     pipelines[name] = pipe

# # Scale-invariant models (no scaler)
# for name, model in non_scaler_models.items():
#     pipe = Pipeline([
#         ('model', model)
#     ])
#     pipe.fit(X_train, y_train)
#     pipelines[name] = pipe


In [ ]:
# for name, pipe in pipelines.items():
#     y_pred = pipe.predict(X_test)
#     score = r2_score(y_test, y_pred)
#     print(f"{name}: R2 Score = {score:.4f}")


In [ ]:
def log_model_result(result_entry: dict, json_file_path: str):
    import json, os

    if not os.path.exists(json_file_path):
        raise FileNotFoundError(f"{json_file_path} not found")

    with open(json_file_path, "r") as f:
        json_data = json.load(f)

    section = result_entry.get("section")
    model_results = result_entry.get("models", {})
    flags = result_entry.get("flags", {})

    # Exit if nothing to log
    if section not in ["modeling", "hyperparameter_tuning"] or not model_results:
        return  # No-op

    if section not in json_data:
        json_data[section] = {}

    for model_name, metrics in model_results.items():
        # If required keys are missing, skip
        if not {"train", "test", "r2_score"}.issubset(metrics.keys()):
            continue

        json_data[section][model_name] = metrics

    # Apply section-level flags if any
    for key, value in flags.items():
        json_data[section][key] = value

    with open(json_file_path, "w") as f:
        json.dump(json_data, f, indent=4)


In [ ]:
# Function to pickle the model and return the file path
def save_model(model, name):
    timestamp = ipp.datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"{name}_{timestamp}.pkl"
    file_path = ipp.os.path.join(ipp.model_dir, file_name)
    
    with open(file_path, "wb") as f:
        ipp.pickle.dump(model, f)

    return file_name  # Returning only the file name to store in JSON

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import os
from sklearn.linear_model import LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV

# === Define models ===
scaler_required_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "SVR": SVR(),
    "KNeighbors": KNeighborsRegressor()
}

non_scaler_models = {
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor()
}

# === Split data ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipelines = {}
model_metrics = {}

# === Train scaler-required models ===
for name, model in scaler_required_models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)

    model_file = save_model(pipe, name)
    
    model_metrics[name] = {
        "train": train_score,
        "test": test_score,
        "r2_score": test_score,
        "model": model_file
    }

# === Train non-scaler models ===
for name, model in non_scaler_models.items():
    pipe = Pipeline([
        ('model', model)
    ])
    pipe.fit(X_train, y_train)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)

    model_file = save_model(pipe, name)

    model_metrics[name] = {
        "train": train_score,
        "test": test_score,
        "r2_score": test_score,
        "model": model_file
    }

# === Prepare structured dict for status.json logging ===
result_entry = {
    "section": "modeling",
    "models": model_metrics,
    "flags": {
        "efficiency": False  # set True manually if needed
    }
}

# === Log to status.json ===
json_file_path = os.path.join("model_dump", "status.json")
log_model_result(result_entry, json_file_path)


In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
json_file_path = os.path.join(ipp.model_dir, "status.json")

with open(json_file_path, "r") as f:
    data = json.load(f)

# Extract models and r2_scores
models = []
r2_scores = []

# From modeling
for model_name, model_info in data.get("modeling", {}).items():
    if isinstance(model_info, dict) and "r2_score" in model_info:
        models.append(model_name)
        r2_scores.append(model_info["r2_score"])

# From hyperparameter_tuning
for model_name, model_info in data.get("hyperparameter_tuning", {}).items():
    if isinstance(model_info, dict) and "r2_score" in model_info:
        models.append(model_name)
        r2_scores.append(model_info["r2_score"])

# Sort models by R² score (ascending)
model_scores = sorted(zip(models, r2_scores), key=lambda x: x[1])
models_sorted, r2_scores_sorted = zip(*model_scores)

# Color palette: last model = light green
base_colors = sns.color_palette("Blues", len(models_sorted)-1)
colors = list(base_colors) + [(0.5, 0.9, 0.5)]  # RGB for light green

# Plot
plt.figure(figsize=(14, 8))
bars = plt.bar(models_sorted, r2_scores_sorted, color=colors, edgecolor="black", alpha=0.7, label="R² Score (Bar)")
plt.plot(models_sorted, r2_scores_sorted, marker="o", color="black", linewidth=2, label="R² Score (Line)")

# Value labels
for bar, score in zip(bars, r2_scores_sorted):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015, f"{score:.3f}",
             ha="center", fontsize=10, fontweight="bold")

# Labels and aesthetics
plt.ylabel("R² Score", fontsize=12)
plt.title("Model and Tuned Model R² Score Comparison", fontsize=15, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Load encrypted content and key from status.json
with open(ipp.json_file_path, "r") as f:
    data = json.load(f)
    encrypted_model_mind = data["Model_Mind"]
    encryption_key = data["key"]

# Decrypt
decrypted_model_mind = ipp.decrypt_model_mind_section(encrypted_model_mind, encryption_key)
print(decrypted_model_mind)